# 16 · Paddy Doctor — bootstrap pareado de P5
**5.000 réplicas · cinco emparejamientos · sin entrenar redes**

El análisis ya se ejecutó y sus resultados se entregan en el paquete Bootstrap 16. Este notebook permite reproducirlo desde el ZIP del notebook 15 guardado en tu Drive.

Selecciona un entorno **CPU, sin GPU**, y pulsa **Ejecutar todas**. No se leen imágenes ni pesos de redes. Solo se utilizan metadatos y predicciones ya calculadas. El tiempo puede variar según Drive y el entorno; el cálculo numérico verificado es pequeño respecto al entrenamiento.

Se estiman intervalos del 95 % por cada emparejamiento. Los cinco comparten imágenes, por lo que no se remuestrean como cinco cohortes independientes ni se calcula un intervalo global basado en esa suposición.


## 1. Conectar Drive y localizar el ZIP
La ruta predeterminada coincide con el experimento 15. Conserva el ZIP original: es el insumo verificable de este análisis.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json, shutil

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/PADDY_DOCTOR_Q1')
INPUT_ZIP = PROJECT / 'exports/15_P5_Rematching_5x3_30ep_v1_RESULTADOS.zip'
OUTPUT = PROJECT / 'results/16_P5_Bootstrap_5000_v1'
REPLICATES = 5000
RANDOM_SEED = 20261121
if not INPUT_ZIP.is_file():
    raise FileNotFoundError(f'Falta el ZIP de las 120 corridas: {INPUT_ZIP}')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Insumo:', INPUT_ZIP)
print('Salida:', OUTPUT)


## 2. Dependencias y código incorporado
Se verifica el ZIP, los archivos de resultados, la firma del plan, los metadatos y las asignaciones. Los pesos entrenados no están en el ZIP y sus bytes no se comprueban aquí. El código completo va incorporado; no necesitas copiarlo de otro archivo.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scikit-learn', 'matplotlib'], check=True)
import importlib.util
MODULE_TEXT = '"""Paired class-stratified bootstrap for the five completed P5 rematchings.\nNo neural-network fitting; no bootstrap of rematchings as independent cohorts.\n"""\nfrom pathlib import Path\nimport argparse,json,hashlib,csv,itertools,zipfile,tempfile,platform,time\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import f1_score,balanced_accuracy_score\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\n\ndef json_write(path,obj):\n    Path(path).write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str)+\'\\n\')\n\n\ndef scores(cm):\n    """Batch confusion matrices (..., K, K) -> macro F1, balanced accuracy, per-class F1/recall."""\n    tp=np.diagonal(cm,axis1=-2,axis2=-1);true=cm.sum(-1);pred=cm.sum(-2)\n    f1=np.divide(2*tp,true+pred,out=np.zeros_like(tp,dtype=float),where=(true+pred)>0)\n    recall=np.divide(tp,true,out=np.zeros_like(tp,dtype=float),where=true>0)\n    return f1.mean(-1),recall.mean(-1),f1,recall\n\n\ndef indicators(y,pred,k):\n    # pred has [architecture, seed, image]. Result [image, architecture*seed*K*K].\n    a,s,n=pred.shape\n    cells=y[None,None,:]*k+pred\n    return np.eye(k*k,dtype=np.float64)[cells].transpose(2,0,1,3).reshape(n,-1)\n\n\ndef confusion_batch(weights,y,pred,k):\n    a,s,n=pred.shape\n    return (weights@indicators(y,pred,k)).reshape(len(weights),a,s,k,k)\n\n\ndef paired_seed_average(values,draws):\n    # values [B, architecture, seed, ...]; draws [B, 3], paired source-control indices.\n    shape=(len(draws),1,draws.shape[1])+(1,)*(values.ndim-3)\n    idx=np.broadcast_to(draws.reshape(shape),(len(draws),values.shape[1],draws.shape[1])+values.shape[3:])\n    return np.take_along_axis(values,idx,axis=2).mean(axis=2)\n\n\ndef interval(values):return np.quantile(values,[.025,.975],axis=0)\n\n\ndef verify_inputs(base):\n    base=Path(base);r=base/\'results\';design=base/\'design\'\n    plan=json.loads((r/\'experiment_plan.json\').read_text());status=json.loads((r/\'completion_summary.json\').read_text())\n    if not status[\'complete\'] or status[\'completed_runs\']!=120 or status[\'evaluations\']!=240:raise ValueError(\'El experimento 15 no está completo\')\n    if sha(r/\'executed_source.py\')!=plan[\'source_sha256\']:raise ValueError(\'Código ejecutado distinto de su huella\')\n    if sha(design/\'03_paddy_doctor_nearclean_frozen.csv\')!=plan[\'metadata_sha256\']:raise ValueError(\'Metadata alterada\')\n    markers=sorted((r/\'runs\').rglob(\'COMPLETE.json\'))\n    if len(markers)!=120:raise ValueError(\'Faltan marcadores de finalización\')\n    checked=0;excluded=0;signatures=set();initializations=[]\n    for marker in markers:\n        record=json.loads(marker.read_text());signatures.add(record[\'signature\'])\n        for entry in record[\'files\']:\n            if entry[\'area\']==\'models\':excluded+=1;continue\n            if entry[\'area\']!=\'results\':raise ValueError(\'Área de manifest desconocida\')\n            p=marker.parent/entry[\'path\']\n            if not p.is_file() or sha(p)!=entry[\'sha256\']:raise ValueError(f\'Archivo alterado o ausente: {p}\')\n            checked+=1\n        prov=json.loads((marker.parent/\'provenance.json\').read_text())\n        key=prov[\'model_key\']\n        if prov[\'tag\']!=plan[\'models\'][key] or prov[\'pretrained\']!=plan[\'pretrained_weights\'][key]:raise ValueError(\'Procedencia de pesos incoherente\')\n        initializations.append({name:prov[name] for name in [\'model_key\',\'train_seed\',\'train_domain\',\'match_seed\',\'initialized_tensor_sha256\']})\n    expected_signature=hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest()\n    if signatures!={expected_signature}:raise ValueError(\'Firma del plan inconsistente\')\n    assignments={}\n    for seed in plan[\'match_seeds\']:\n        p=design/f\'match_{seed}\'/\'assignments.csv\'\n        if sha(p)!=plan[\'assignments\'][str(seed)]:raise ValueError(\'Asignaciones alteradas\')\n        d=pd.read_csv(p,dtype={\'image_id\':str});assignments[seed]=d\n        if len(d)!=1000 or d.image_id.nunique()!=1000:raise ValueError(\'Cohorte no contiene 1000 IDs únicos\')\n        for split,n in [(\'train\',60),(\'validation\',20),(\'test\',20)]:\n            sub=d[d.pair_split.eq(split)]\n            counts=sub.groupby([\'label\',\'age\',\'domain\']).size().unstack(fill_value=0)\n            if not counts.ADT45.eq(counts.OTHER).all():raise ValueError(\'Emparejamiento de edad incorrecto\')\n            if not sub.groupby([\'domain\',\'label\']).size().eq(n).all():raise ValueError(\'Tamaño de partición incorrecto\')\n    verification=dict(completed_runs=120,evaluations=240,result_files_sha256_verified=checked,\n        checkpoint_hashes_recorded_but_weights_not_in_results_zip=excluded,executed_source_hash_verified=True,\n        metadata_hash_verified=True,assignment_hashes_verified=5,plan_signature_verified=True,\n        note=\'Checkpoint payloads were not supplied in the results ZIP; their bytes were not independently checked here.\')\n    return plan,assignments,verification,pd.DataFrame(initializations)\n\n\ndef load_predictions(base,plan,assignments,match,target):\n    root=Path(base)/\'results/runs\'/f\'match_{match}\';models=list(plan[\'models\']);seeds=plan[\'train_seeds\'];classes=plan[\'class_names\']\n    expected=assignments[match].query(\'domain == @target and pair_split == "test"\').sort_values(\'image_id\').reset_index(drop=True)\n    ids=expected.image_id.tolist();y=expected.label.map({x:i for i,x in enumerate(classes)}).to_numpy()\n    if len(ids)!=100 or sorted(np.bincount(y).tolist())!=[20]*5:raise ValueError(\'Test no es 20 por clase\')\n    pc=[];pt=[];max_error=0.;checks=0\n    for key in models:\n        c=[];t=[]\n        for seed in seeds:\n            for domain,container in [(target,c),(\'OTHER\' if target==\'ADT45\' else \'ADT45\',t)]:\n                folder=root/f\'train_{domain}__{key}__seed_{seed}\'/f\'eval__{target}\'\n                d=pd.read_csv(folder/\'predictions.csv\',dtype={\'image_id\':str}).sort_values(\'image_id\').reset_index(drop=True)\n                if d.image_id.tolist()!=ids:raise ValueError(\'Control y transferencia no usan los mismos IDs de test\')\n                for col in [\'label\',\'age\',\'domain\',\'sha256\',\'matched_pair_id\']:\n                    if not np.array_equal(d[col].astype(str),expected[col].astype(str)):raise ValueError(f\'Metadata de test no coincide: {col}\')\n                pred=d.y_pred_raw.map({x:i for i,x in enumerate(classes)}).to_numpy()\n                if pd.isna(pred).any() or not np.array_equal(d.y_true_idx.to_numpy(),y):raise ValueError(\'Orden de clases incorrecto\')\n                archived=json.loads((folder/\'metrics.json\').read_text())[\'raw\']\n                err=abs(f1_score(y,pred,labels=range(5),average=\'macro\',zero_division=0)-archived[\'macro_f1\'])\n                if err>1e-12:raise ValueError(\'Macro-F1 no reproduce el archivo\')\n                max_error=max(max_error,err);checks+=1;container.append(pred)\n        pc.append(c);pt.append(t)\n    return expected,y,np.asarray(pc),np.asarray(pt),max_error,checks\n\n\ndef analyze(base,output,replicates=5000,random_seed=20261121):\n    if replicates<1000:raise ValueError(\'Usa al menos 1000 réplicas para el análisis entregable\')\n    start=time.perf_counter();base=Path(base);output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    plan,assignments,verification,init=verify_inputs(base)\n    init.to_csv(output/\'initialized_weight_provenance.csv\',index=False)\n    models=list(plan[\'models\']);seeds=plan[\'train_seeds\'];classes=plan[\'class_names\'];k=len(classes)\n    rng=np.random.default_rng(random_seed);macro_rows=[];class_rows=[];seed_rows=[];replicate_payload={};max_error=0.;checks=0\n    for match in plan[\'match_seeds\']:\n        for target in [\'ADT45\',\'OTHER\']:\n            metadata,y,pc,pt,err,nchecks=load_predictions(base,plan,assignments,match,target)\n            max_error=max(max_error,err);checks+=nchecks;n=len(y)\n            # Draw exactly 20 image indices per true class, shared by all architectures and seeds.\n            weights=np.zeros((replicates,n),dtype=np.float64)\n            for label in range(k):\n                positions=np.flatnonzero(y==label)\n                weights[:,positions]=rng.multinomial(len(positions),np.full(len(positions),1/len(positions)),size=replicates)\n            draws=rng.integers(0,len(seeds),size=(replicates,len(seeds)))\n            control_cm=confusion_batch(weights,y,pc,k);transfer_cm=confusion_batch(weights,y,pt,k)\n            point_c=scores(confusion_batch(np.ones((1,n)),y,pc,k));point_t=scores(confusion_batch(np.ones((1,n)),y,pt,k))\n            boot_c=scores(control_cm);boot_t=scores(transfer_cm)\n            prefix=f\'M{match}_{target}\'\n            for metric_index,metric in [(0,\'macro_f1\'),(1,\'balanced_accuracy\')]:\n                # Mean of seed-specific nonlinear scores; never compute F1 after pooling seeds.\n                differences=boot_c[metric_index]-boot_t[metric_index]\n                fixed=differences.mean(axis=2);hierarchical=paired_seed_average(differences,draws)\n                point=(point_c[metric_index]-point_t[metric_index])[0]\n                fixed_ci=interval(fixed);hier_ci=interval(hierarchical)\n                replicate_payload[f\'{prefix}_{metric}_image\']=fixed\n                replicate_payload[f\'{prefix}_{metric}_seed_image\']=hierarchical\n                for a,key in enumerate(models):\n                    macro_rows.append(dict(match_seed=match,target=target,model_key=key,metric=metric,n_target_images=n,\n                        n_training_seeds=len(seeds),control_score=float(point_c[metric_index][0,a].mean()),\n                        transfer_score=float(point_t[metric_index][0,a].mean()),loss=float(point[a].mean()),\n                        image_ci95_lower=float(fixed_ci[0,a]),image_ci95_upper=float(fixed_ci[1,a]),\n                        seed_image_interval95_lower=float(hier_ci[0,a]),seed_image_interval95_upper=float(hier_ci[1,a]),\n                        image_bootstrap_nonpositive_fraction=float(np.mean(fixed[:,a]<=0)),\n                        seed_image_nonpositive_fraction=float(np.mean(hierarchical[:,a]<=0))))\n                    for s,seed in enumerate(seeds):\n                        low,high=interval(differences[:,a,s])\n                        seed_rows.append(dict(match_seed=match,target=target,model_key=key,train_seed=seed,metric=metric,\n                            loss=float(point[a,s]),image_ci95_lower=float(low),image_ci95_upper=float(high)))\n            for metric_index,metric in [(2,\'class_f1\'),(3,\'class_recall\')]:\n                dif=boot_c[metric_index]-boot_t[metric_index]\n                fixed=dif.mean(2);hierarchical=paired_seed_average(dif,draws)\n                fixed_ci=interval(fixed);hier_ci=interval(hierarchical)\n                cci=interval(boot_c[metric_index].mean(2));tci=interval(boot_t[metric_index].mean(2))\n                for a,key in enumerate(models):\n                    for label,name in enumerate(classes):\n                        class_rows.append(dict(match_seed=match,target=target,model_key=key,class_name=name,metric=metric,\n                            n_unique_class_images=int((y==label).sum()),n_training_seeds=3,\n                            control_score=float(point_c[metric_index][0,a,:,label].mean()),\n                            transfer_score=float(point_t[metric_index][0,a,:,label].mean()),\n                            control_image_ci95_lower=float(cci[0,a,label]),control_image_ci95_upper=float(cci[1,a,label]),\n                            transfer_image_ci95_lower=float(tci[0,a,label]),transfer_image_ci95_upper=float(tci[1,a,label]),\n                            loss=float((point_c[metric_index]-point_t[metric_index])[0,a,:,label].mean()),\n                            image_ci95_lower=float(fixed_ci[0,a,label]),image_ci95_upper=float(fixed_ci[1,a,label]),\n                            seed_image_interval95_lower=float(hier_ci[0,a,label]),seed_image_interval95_upper=float(hier_ci[1,a,label])))\n            # Cross-check the vectorized confusion/F1 implementation with independent sklearn calls.\n            for a in range(len(models)):\n                for s in range(3):\n                    for b in [0,replicates//2,replicates-1]:\n                        for pred,score_batch in [(pc,boot_c),(pt,boot_t)]:\n                            independent=f1_score(y,pred[a,s],labels=range(k),average=\'macro\',sample_weight=weights[b],zero_division=0)\n                            diff=abs(independent-score_batch[0][b,a,s]);max_error=max(max_error,diff);checks+=1\n                            if diff>1e-12:raise ValueError(\'Bootstrap vectorizado no coincide con sklearn\')\n            print(f\'Bootstrap terminado: matching {match}, objetivo {target}\',flush=True)\n    macro=pd.DataFrame(macro_rows);perclass=pd.DataFrame(class_rows);perseed=pd.DataFrame(seed_rows)\n    macro.to_csv(output/\'16_paired_bootstrap_by_matching.csv\',index=False)\n    perclass.to_csv(output/\'16_class_f1_recall_bootstrap.csv\',index=False)\n    perseed.to_csv(output/\'16_per_seed_paired_bootstrap.csv\',index=False)\n    np.savez_compressed(output/\'16_macro_bootstrap_replicates.npz\',**replicate_payload)\n    # Reconstruct and verify all 120 original seed-level macro-F1 losses.\n    original=pd.read_csv(base/\'results/fixed_target_losses_by_seed.csv\')\n    joined=perseed[perseed.metric.eq(\'macro_f1\')].merge(original,on=[\'match_seed\',\'target\',\'model_key\',\'train_seed\'],validate=\'one_to_one\',suffixes=(\'_new\',\'_original\'))\n    if len(joined)!=120 or not np.allclose(joined.loss_new,joined.loss_original,atol=1e-12,rtol=0):raise ValueError(\'No se reproducen las 120 pérdidas originales\')\n    global_summary=macro.groupby([\'metric\',\'model_key\',\'target\']).agg(n_rematchings=(\'match_seed\',\'nunique\'),\n        mean_loss=(\'loss\',\'mean\'),sd_between_rematchings=(\'loss\',\'std\'),min_loss=(\'loss\',\'min\'),max_loss=(\'loss\',\'max\'),\n        n_image_intervals_above_zero=(\'image_ci95_lower\',lambda v:int((v>0).sum())),\n        n_seed_image_intervals_above_zero=(\'seed_image_interval95_lower\',lambda v:int((v>0).sum()))).reset_index()\n    global_summary.to_csv(output/\'16_descriptive_across_matching_summary.csv\',index=False)\n    overlap=pd.read_csv(base/\'design/overlap_between_cohorts.csv\')\n    overlap.to_csv(output/\'16_overlap_between_cohorts.csv\',index=False)\n    verification.update(dict(numerical_checks=checks,max_absolute_discrepancy=max_error,original_seed_losses_reproduced=120,\n        macro_comparisons=40,matched_stratified_draws=True,replicates_per_comparison=replicates,\n        neural_network_training_performed=False,elapsed_seconds=time.perf_counter()-start))\n    json_write(output/\'16_verification.json\',verification)\n    config=dict(replicates=replicates,random_seed=random_seed,interval_method=\'percentile 2.5% and 97.5%; not BCa\',\n        estimand=\'control minus transfer score on identical target images; average of three seed-specific scores\',\n        image_bootstrap=\'100 test images sampled within true class (20 per class), with replacement; same indices for control, transfer, all architectures and seeds within a matching/target\',\n        seed_image_sensitivity=\'same class-stratified image draws plus resampling three paired seed indices with replacement; shared indices across architectures\',\n        rematching_handling=\'five matching constructions analyzed separately; descriptive mean/SD/range only across constructions; no resampling of overlapping constructions as independent datasets\',\n        limits=[\'conditional on fitted networks, selected checkpoints and frozen matching assignments\',\n                \'three seeds yield a limited exploratory seed-distribution sensitivity, not precise seed-population coverage\',\n                \'no simultaneous coverage or multiplicity correction; nonpositive bootstrap fractions are not frequentist p-values\',\n                \'no refitting, farm/session/plant cluster bootstrap or external-domain uncertainty\',\n                \'no BCa correction; class F1/recall intervals have only 20 unique class images and are exploratory\',\n                \'tests had been inspected before this retrospective analysis\'],\n        models=models,classes=classes,match_seeds=plan[\'match_seeds\'],train_seeds=seeds,\n        python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__)\n    json_write(output/\'16_bootstrap_config.json\',config)\n    report(output,macro,global_summary,verification,config)\n    figures(output,macro,models,plan[\'match_seeds\'],replicates)\n    checksum(output)\n    return macro,global_summary\n\n\ndef report(out,macro,summary,verification,config):\n    f=macro[macro.metric.eq(\'macro_f1\')];s=summary[summary.metric.eq(\'macro_f1\')]\n    lines=[\'# Bootstrap pareado — cinco emparejamientos P5\',\'\',\n        f\'Se ejecutaron {config["replicates"]:,} réplicas por comparación, sin entrenar redes ni utilizar GPU. Se conservaron las 120 corridas y 240 evaluaciones del notebook 15.\',\n        \'\', \'## Resultado principal\',\'\',\n        f\'Las {len(f)} pérdidas medias de macro-F1 son positivas. Los intervalos por imágenes excluyen cero en {int((f.image_ci95_lower>0).sum())}/{len(f)} comparaciones. Los intervalos de sensibilidad por semillas e imágenes excluyen cero en {int((f.seed_image_interval95_lower>0).sum())}/{len(f)}.\',\n        \'\', \'Estos son intervalos marginales por emparejamiento y configuración; el conjunto no tiene una garantía simultánea del 95 %. El recuento no se interpreta como un p-valor global.\',\n        \'\', \'| Configuración | Objetivo | Pérdida media entre cinco construcciones | Rango de pérdidas | IC por imágenes sobre cero | Sensibilidad semillas e imágenes sobre cero |\',\n        \'| --- | --- | ---: | --- | ---: | ---: |\']\n    for r in s.itertuples():\n        lines.append(f\'| {r.model_key} | {r.target} | {r.mean_loss:.4f} | {r.min_loss:.4f}–{r.max_loss:.4f} | {r.n_image_intervals_above_zero}/5 | {r.n_seed_image_intervals_above_zero}/5 |\')\n    lines+=[\'\',\'## Qué se remuestreó\',\'\',\n        \'En cada construcción y dominio objetivo se seleccionan 20 imágenes con reemplazo dentro de cada una de las cinco clases. Los mismos índices se aplican a control y transferencia, a las cuatro arquitecturas y a las tres semillas. Se calcula macro-F1 por semilla y después se promedian los tres resultados; no se agrupan 300 predicciones como si fueran 300 imágenes independientes.\',\n        \'\', \'El análisis adicional remuestrea tres índices de semilla emparejados con reemplazo y las mismas imágenes. Con solo tres semillas, ese intervalo es una sensibilidad exploratoria. No implica reentrenar redes ni estimar con precisión toda la variación posible de entrenamiento.\',\n        \'\', \'Los cinco emparejamientos comparten imágenes. Por eso no se remuestrean como cinco cohortes independientes ni se obtiene un IC global que presuponga su independencia. La media, desviación y rango entre construcciones se presentan descriptivamente. Los intervalos no cuantifican nuevas fincas, otra adquisición, nuevas asignaciones de entrenamiento ni variación de un nuevo ajuste.\',\n        \'\', \'## Verificación\',\'\',\n        f\'Se verificaron las huellas de {verification["result_files_sha256_verified"]} archivos de resultados, las cinco asignaciones, los metadatos y el código ejecutado. Las 120 pérdidas originales se reprodujeron. Se efectuaron {verification["numerical_checks"]} comprobaciones numéricas con discrepancia máxima {verification["max_absolute_discrepancy"]:.3g}. Los archivos de pesos entrenados no estaban en el ZIP; sus bytes no se verificaron aquí.\',\n        \'\', \'## Texto propuesto para el manuscrito\',\'\',\n        \'**Methods.** Within each of the five frozen matching constructions and each target domain, we generated 5,000 paired class-stratified bootstrap samples of the 100 target-test images, retaining 20 observations per true class. The same sampled image indices were applied to control and transferred predictions across all architectures and three training seeds. The statistic was the mean of seed-specific control-minus-transfer macro-F1 differences. Marginal percentile 95% intervals used the 2.5th and 97.5th percentiles. A separately labeled exploratory sensitivity additionally resampled three paired seed indices with replacement. Matching constructions were not resampled as independent cohorts because their image sets overlapped; between-construction variation was summarized descriptively. Models, checkpoints and matching assignments remained fixed.\',\n        \'\', f\'**Results.** Positive fixed-target macro-F1 losses persisted across all 40 architecture–target–matching configurations. Image-conditional intervals excluded zero in {int((f.image_ci95_lower>0).sum())} of 40 comparisons; the seed-and-image sensitivity intervals excluded zero in {int((f.seed_image_interval95_lower>0).sum())} of 40. These marginal retrospective intervals do not establish simultaneous coverage, uncertainty under rematching or external-domain generalization.\',\n        \'\', \'## Archivos\',\'\',\n        \'- `16_paired_bootstrap_by_matching.csv`: 40 contrastes macro-F1 y 40 de exactitud balanceada, con ambos intervalos.\',\n        \'- `16_per_seed_paired_bootstrap.csv`: intervalos por semilla, manteniendo el par control/transferencia.\',\n        \'- `16_class_f1_recall_bootstrap.csv`: F1 y recall por clase, media de las tres semillas; intervalos exploratorios.\',\n        \'- `16_descriptive_across_matching_summary.csv`: resumen descriptivo entre construcciones, sin IC global de independencia.\',\n        \'- `16_macro_bootstrap_replicates.npz`: réplicas para trazabilidad; ejes réplicas × arquitectura, orden en configuración.\',\n        \'- Figuras PNG/PDF, configuración, verificación e inventario SHA-256.\']\n    (out/\'16_Informe_Bootstrap_P5.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\n\n\ndef figures(out,macro,models,matches,replicates=5000):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'savefig.dpi\':300})\n    display={\'resnet50\':(\'ResNet-50\',\'#216C92\'),\'convnext_tiny\':(\'ConvNeXt-Tiny\',\'#CC7B26\'),\n             \'vit_small_patch16_224\':(\'ViT-S/16\',\'#27856D\'),\'swin_tiny_patch4_window7_224\':(\'Swin-Tiny\',\'#AE4962\')}\n    models=[key for key in display if key in models]\n    # Each marker is one matching, not a single pooled effect.\n    for mode,lo,hi,label in [(\'image\',\'image_ci95_lower\',\'image_ci95_upper\',\'Conditional image bootstrap\'),\n                            (\'seed_image\',\'seed_image_interval95_lower\',\'seed_image_interval95_upper\',\'Seed-and-image sensitivity\')]:\n        fig,axes=plt.subplots(1,2,figsize=(11,7),sharex=True,sharey=True)\n        for ax,target in zip(axes,[\'ADT45\',\'OTHER\']):\n            for a,key in enumerate(models):\n                sub=macro[(macro.metric==\'macro_f1\')&(macro.target==target)&(macro.model_key==key)].set_index(\'match_seed\').loc[matches]\n                y=a*6+np.arange(5)\n                ax.errorbar(sub.loss,y,xerr=np.vstack([sub.loss-sub[lo],sub[hi]-sub.loss]),fmt=\'o\',color=display[key][1],capsize=2,ms=4,lw=1.2)\n            ax.axvline(0,ls=\'--\',color=\'.4\',lw=1);ax.grid(axis=\'x\',alpha=.2);ax.set_title(\'Target \'+target)\n            ax.set_xlabel(\'Control − transfer macro-F1\')\n        ticks=[a*6+i for a in range(4) for i in range(5)]\n        axes[0].set_yticks(ticks,[f\'{display[models[a]][0]} · M{i+1}\' for a in range(4) for i in range(5)])\n        axes[0].invert_yaxis();fig.suptitle(label+f\' · {replicates:,} paired replicates\');fig.tight_layout()\n        for ext in [\'png\',\'pdf\']:fig.savefig(out/f\'16_{mode}_intervals.{ext}\',bbox_inches=\'tight\')\n        plt.close(fig)\n\n\ndef checksum(out):\n    with (Path(out)/\'SHA256.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'relative_path\',\'size_bytes\',\'sha256\'])\n        for p in sorted(Path(out).rglob(\'*\')):\n            if p.is_file() and p.name!=\'SHA256.csv\':w.writerow([p.relative_to(out).as_posix(),p.stat().st_size,sha(p)])\n\n\ndef run_from_zip(input_zip,output,replicates=5000,random_seed=20261121):\n    with tempfile.TemporaryDirectory(prefix=\'paddy16_\') as temp:\n        with zipfile.ZipFile(input_zip) as archive:\n            for item in archive.infolist():\n                dest=(Path(temp)/item.filename).resolve()\n                if not dest.is_relative_to(Path(temp).resolve()):raise ValueError(\'Ruta de ZIP no segura\')\n            archive.extractall(temp)\n        result=analyze(Path(temp),output,replicates,random_seed)\n    json_write(Path(output)/\'16_input_archive.json\',dict(file_name=Path(input_zip).name,size_bytes=Path(input_zip).stat().st_size,sha256=sha(input_zip)))\n    checksum(output)\n    return result\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--input-zip\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--replicates\',type=int,default=5000);parser.add_argument(\'--seed\',type=int,default=20261121)\n    args=parser.parse_args();run_from_zip(args.input_zip,args.output,args.replicates,args.seed)\n'
MODULE_PATH = Path('/content/paddy16_bootstrap.py')
MODULE_PATH.write_text(MODULE_TEXT, encoding='utf-8')
spec = importlib.util.spec_from_file_location('paddy16_bootstrap', MODULE_PATH)
p16 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = p16
spec.loader.exec_module(p16)


## 3. Ejecutar el bootstrap
En cada construcción y objetivo:

- Se remuestrean las 100 imágenes dentro de clase: 20 por cada una de cinco clases.
- Se aplican los mismos índices a control y transferencia, a las cuatro arquitecturas y a las tres semillas.
- Se calcula F1 por semilla y después su promedio; no se juntan las tres semillas como 300 imágenes independientes.
- El segundo intervalo añade remuestreo de los tres índices de semillas emparejadas. Con tres semillas se interpreta como **sensibilidad exploratoria**.

Se usan percentiles 2,5 y 97,5; **no es BCa**. Los intervalos son marginales, sin corrección por comparaciones múltiples. Las fracciones de réplicas no positivas no son p-valores frecuentistas. Los modelos, checkpoints y asignaciones permanecen fijos.


In [ ]:
macro, across = p16.run_from_zip(INPUT_ZIP, OUTPUT,
                                replicates=REPLICATES, random_seed=RANDOM_SEED)
# Conservar el código exacto utilizado junto a los resultados.
shutil.copyfile(MODULE_PATH, OUTPUT / 'paddy16_bootstrap.py')
p16.checksum(OUTPUT)
print('Bootstrap terminado, sin entrenar redes.')


## 4. Revisar las tablas y figuras
`loss` es el F1 de control menos el F1 transferido sobre **las mismas imágenes objetivo**. Un límite inferior mayor que cero indica una pérdida positiva en ese intervalo marginal. La tabla entre construcciones es descriptiva; no supone que los cinco emparejamientos sean independientes.


In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
primary = macro.query("metric == 'macro_f1'")
display(primary[['match_seed','target','model_key','loss',
                 'image_ci95_lower','image_ci95_upper',
                 'seed_image_interval95_lower','seed_image_interval95_upper']])
display(across.query("metric == 'macro_f1'"))
display(Image(filename=str(OUTPUT / '16_image_intervals.png')))
display(Image(filename=str(OUTPUT / '16_seed_image_intervals.png')))
display(Markdown((OUTPUT / '16_Informe_Bootstrap_P5.md').read_text()))


## 5. Exportar
El paquete de salida contiene tablas, réplicas, figuras, configuración, verificación y código. El ZIP original de entrenamiento se conserva aparte. Para actualizar el manuscrito se necesitan ambos archivos.


In [ ]:
EXPORT = PROJECT / 'exports/16_P5_Bootstrap_5000_RESULTADOS'
zip_path = shutil.make_archive(str(EXPORT), 'zip', root_dir=OUTPUT)
print('Archivo creado:', zip_path)
print('No se ejecutó ningún entrenamiento adicional.')
# Descarga opcional al ordenador:
# from google.colab import files
# files.download(zip_path)


### Alcance de la evidencia
Los intervalos condicionan en las imágenes y modelos disponibles y no sustituyen validación externa, independencia por finca/planta/sesión ni nuevos ajustes. El test ya se había examinado antes de este análisis retrospectivo. La sensibilidad con tres semillas no garantiza cobertura de toda la población de posibles entrenamientos.
